In [ ]:
# using content/FE_train.csv to train random survival forest model and then generate predictions using content/FE_test.csv
import pandas as pd
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# load training data
train = pd.read_csv('content/FE_train.csv')
train = train.drop(columns=['event_id']) # we don't want to use event_id as a feature for prediction
#print(train["time_to_hit_hours"].max()) # gives us 66.99447413277778


In [6]:
# create survival object
y = Surv.from_dataframe('event', 'time_to_hit_hours', train) #specifying column names that represent event and time to event
X = train.drop(columns=['event', 'time_to_hit_hours']) # and then we don't want to train with those columns so dropping them

In [7]:
# create RSF model
rsf = RandomSurvivalForest(
    n_estimators=100, 
    min_samples_split=10, 
    min_samples_leaf=15, 
    max_features="sqrt", 
    n_jobs=-1, 
    random_state=42
) # can play around with each of these values later to see what leads to the best performance

# fit model
rsf.fit(X, y)

,n_estimators,100
,max_depth,None
,min_samples_split,10
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,42


In [ ]:
# using content/FE_test.csv  
test = pd.read_csv('content/FE_test.csv')
X_test = test.drop(columns=['event_id']) # we don't want to use event_id as a feature for prediction
test_survival_predictions = rsf.predict_survival_function(X_test) # this will give us the predicted survival function for each observation in our training data
# aka gives us probability that fire i has NOT hit by time t
# so the probability that fire i HAS hit is 1 - survival_predictions[i](t) for each time t

In [9]:
times = [12, 24, 48, 72] # the hours we need to calculate the probability of wildfire hitting within

max_time = train["time_to_hit_hours"].max() # gives us 66.99447413277778
# this lets us know that we have no data on wildfires hitting after 66.99447413277778 hours, so we can only calculate probabilities for times up to that point
# meaning that we can't directly calculate probabilities for 72 hours, but we can still calculate for 12, 24, and 48 hours since those are all less than the max time in our training data
# will need to handle 72 hour time 
# for now, instead of calculating probability for 72 hours, calculate the probability for 66.99... hours and use that as our estimate instead

test_probabilities = []
for fn in test_survival_predictions:
    prob = []
    for t in times:
        t = min(t, max_time) # if t is greater than max_time, use max_time instead
        prob.append(1 - fn(t)) 
    test_probabilities.append(prob)
print(test_probabilities)

[[np.float64(0.08899731887699147), np.float64(0.15920597684667737), np.float64(0.1737979898401245), np.float64(0.2970323732655126)], [np.float64(0.2629919696898402), np.float64(0.45233143826078337), np.float64(0.4748681066757334), np.float64(0.5712327373580299)], [np.float64(0.08762758395361747), np.float64(0.15243702097815015), np.float64(0.17196980972924092), np.float64(0.2909747431189764)], [np.float64(0.2753314414567172), np.float64(0.46731260490327475), np.float64(0.48797672458530106), np.float64(0.6169341579225893)], [np.float64(0.618769879637501), np.float64(0.6295886130553006), np.float64(0.6372986423639473), np.float64(0.6433141204720372)], [np.float64(0.06408571061354151), np.float64(0.0983341219245949), np.float64(0.10518487299361556), np.float64(0.16286347078751195)], [np.float64(0.09058956746004898), np.float64(0.1557051681825753), np.float64(0.1763242301290604), np.float64(0.292133213799727)], [np.float64(0.23286707592018285), np.float64(0.4005201185120745), np.float64(0.

In [10]:
# doing checks on test_probabilities to make sure they look reasonable
print(len(test_probabilities) == len(test)) # just making sure we have a probability for each row in test data
# check that all probabilities are between 0 and 1
for prob in test_probabilities:
    for p in prob:
        if p < 0 or p > 1:
            print("Probability out of bounds:", p)
# making sure monotonicity enforced row-wise (prob_12h <= prob_24h <= prob_48h <= prob_72h)
for prob in test_probabilities:
    for i in range(1, len(prob)):
        if prob[i] < prob[i-1]:
            print("Monotonicity violated:", prob)

True


In [11]:
# now we want to take our probabilities and put it into CSV with one row per event_id and four probability columns: event_id, prob_12h, prob_24h, prob_48h, prob_72h
output_df = pd.DataFrame()
output_df['event_id'] = test['event_id']
output_df['prob_12h'] = [p[0] for p in test_probabilities]
output_df['prob_24h'] = [p[1] for p in test_probabilities]
output_df['prob_48h'] = [p[2] for p in test_probabilities]
output_df['prob_72h'] = [p[3] for p in test_probabilities]
output_df.to_csv('content/submission2.csv', index=False)